# Deep Learning for Engineers: Exercise 1

This exercise serves as an introduction to deep learning concepts, focusing on building and training neural networks for image classification.

In this project, we have a dataset consisting of hand-drawn shapes:

- `training_data/circles/`: contains 100 images of circles

- `training_data/triangles/`: contains 100 images of triangles

Our goal is to train a neural network that can accurately distinguish between circles and triangles. After training, we will evaluate the model by using it to classify new, unseen images stored in the test_images folder.

Through this exercise, you will gain hands-on experience in:

- Preprocessing image data

- Defining and training a neural network

- Evaluating model performance

- Making predictions on real input data

**Parts of the code where you are expected to write or modify something are marked with comments starting with @Student.**

#### Importing Libraries and Modules

In this part, several important Python libraries and modules are imported to assist with the task of building and training a neural network for image classification:

- **numpy**: This library is essential for working with arrays and matrices, which are fundamental data structures for numerical computations in machine learning and computer vision.

- **os**: This module allows us to interact with the operating system, enabling tasks such as file path manipulation and directory creation.

- **cv2**: The **OpenCV** library is used for image processing tasks, such as reading images, resizing, and applying transformations.

- **torch**: The **PyTorch** library is the core deep learning framework used in this exercise. It provides the necessary tools for building and training neural networks, such as tensors (multi-dimensional arrays), loss functions, and optimization techniques.

- **torch.nn**: This module contains pre-built layers and models for constructing neural networks, including **fully connected layers**, **activation functions**, and **loss functions**.

- **torch.optim**: The **optim** module provides optimization algorithms, such as **Adam**, which are used to update the weights of the neural network during training.

- **time**: This is a standard Python module used to measure time-related functions.

- **utils**: This custom module imports functions like `preprocess_test_image` and `preprocess_training_images`, which help in preparing the image data by resizing and normalizing it to fit the input requirements of the neural network.

By using these libraries and functions, we can efficiently handle data preprocessing, build and train deep learning models, and evaluate their performance.


In [1]:
# @Student: Import the numpy library using the alias "np".
import numpy as np
import os
import cv2 as cv
import torch
import torch.nn as nn
import torch.optim as optim
import time
from utils import preprocess_test_image, preprocess_training_images, IMAGE_SIZE

Now, we will set up the file path structure for the project by using the current working directory as a base. The os.getcwd() function returns the current working directory, which is stored in home_dir.

Then, using os.path.join(), the script constructs full paths to various subdirectories where data is stored:

- **path_to_test_images**: Directory containing test images used for evaluating the model.

- **path_to_training_data**: Main directory containing the training data.

- **path_to_model**: Directory where the trained model will be saved.

- **path_to_circles** and **path_to_triangles**: Subdirectories containing training images of circles and triangles, respectively.

In [2]:
home_dir = os.getcwd()

path_to_test_images = os.path.join(home_dir, 'test_images')
path_to_training_data = os.path.join(home_dir, 'training_data')
path_to_model = os.path.join(home_dir, 'results')
# @Student: Complete the code by defining path_to_circles and path_to_triangles using os.path.join().
# Hint: These folders are inside the training_data directory.
path_to_circles = os.path.join(path_to_training_data, 'circles')
path_to_triangles = os.path.join(path_to_training_data, 'triangles')

#### Preparing the Training Data and Labels

This section loads image files, processes them, and prepares the dataset for training.

First, image file paths are collected. The following lines generate lists of full file paths for all images in the circles and triangles folders.

In [3]:
circles = [os.path.join(path_to_circles, f) for f in os.listdir(path_to_circles)]
triangles = [os.path.join(path_to_triangles, f) for f in os.listdir(path_to_triangles)]

Then, the data arrays are initialized.

- **`training_data`**: A 2D NumPy array where each row contains one image, flattened into a 1D array. This will be used as input for the neural network.

- **`labels`**: A 1D array where each element is an integer representing the class label:

  - `[1]` → **Circle**
  - `[0]` → **Triangle**

In [4]:
training_data = np.zeros((len(circles) + len(triangles), IMAGE_SIZE * IMAGE_SIZE), dtype=np.float32)
labels = np.zeros(len(circles) + len(triangles), dtype=np.int64)

Next, the images are processed and stored with their labels.

In [5]:
for i, path in enumerate(circles):
    processed_image = preprocess_training_images(path)
    training_data[i] = processed_image.flatten()
    # @Student: Complete the code by assigning the correct label for the circles.
    labels[i] = 1

for i, path in enumerate(triangles):
    # @Student: Preprocess the triangle image using the preprocess_training_images() function.
    processed_image = preprocess_training_images(path)
    training_data[len(circles) + i] = processed_image.flatten()
    # @Student: Complete the code by assigning the correct label for the triangles.
    # Hint: The label index for triangles will start from len(circles).
    labels[len(circles) + i] = 0

The training data and their corresponding labels are randomly shuffled. This ensures that the model sees a good mix of different classes (in this case, circles and triangles) during each training batch. This prevents the model from learning patterns based on the order of the data, which could lead to bias or poor generalization. Without shuffling, the model might train on only one class at a time (e.g., all circles first), making it harder to learn to distinguish between different shapes.


In [6]:
indices = np.arange(len(training_data))
np.random.shuffle(indices)
training_data = training_data[indices]
labels = labels[indices]

The NumPy arrays `training_data` and `labels` are converted into PyTorch tensors `X` and `y`, which are the required data types for training a model in PyTorch.

In [7]:
X = torch.tensor(training_data)
y = torch.tensor(labels)

#### Defining the Model Architecture

This section defines the neural network structure. You can experiment with changing the number of layers, the number of neurons in each layer, and the activation functions to see how model complexity affects training and performance.

In [8]:
# @Student: Modify the architecture below to try out different model designs.
model = nn.Sequential(
    nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 512),
    nn.ReLU(),
    nn.Linear(512, 2)
)

### Very few neurons:
#model = nn.Sequential(
#    nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 4),
#    nn.ReLU(),
#    nn.Linear(4, 2)
#)

### A very large network:
#model = nn.Sequential(
#    nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 1024),
#    nn.ReLU(),
#    nn.Linear(1024, 512),
#    nn.ReLU(),
#    nn.Linear(512, 256),
#    nn.ReLU(),
#    nn.Linear(256, 2)
#)


#### Training Configuration

Here we define training hyperparameters such as number of epochs, batch size, loss function, and optimizer. Try changing the values of epochs and batch size to see how training speed and accuracy are affected.

- **nn.CrossEntropyLoss()**: It's a loss function used in classification problems. It compares the model's output (raw scores called logits - not probabilities) with the true class label (an integer like 0, 1, or 2), and it computes how wrong the prediction is - giving a single number (the loss) that we try to minimize during training.

- **optim.Adam**: Adam (Adaptive Moment Estimation) is one of the most popular optimization algorithms in deep learning. It adjusts how much to change each weight and bias in the model based on:

  - Gradient (first moment): Measures how steep the error slope is.

  - Momentum (second moment): Tracks the smoothness and consistency of the gradient over time.

  - Adaptive Learning Rate: Adjusts the learning rate dynamically for each parameter based on its gradient history.



In [9]:
# @Student: Try different values of EPOCHS and BATCH_SIZE to observe their effect on training.
EPOCHS = 20 # Try values like 5, 50, 100
BATCH_SIZE = 10 # Try values like 1, 20, len(X)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

#### Training Loop

This is the core training loop. In each epoch, the model is trained on all batches of the dataset. You can monitor the loss value to track the learning process.

In [10]:
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    total_loss = 0
    total_correct = 0
    total_samples = 0
    num_batches = 0

    for i in range(0, len(X), BATCH_SIZE):
        # Slice out a mini-batch of inputs and corresponding targets:
        input_img_batch = X[i:i + BATCH_SIZE]
        target_batch = y[i:i + BATCH_SIZE] # @Student: Get the corresponding target labels.

        # Get predictions (logits) from the model:
        output_batch = model(input_img_batch)

        # Compare model predictions to ground truth using CrossEntropyLoss:
        loss = loss_fn(output_batch, target_batch)

        # Backpropagation:
        optimizer.zero_grad() # Clear previous gradients.
        loss.backward() # Compute gradients.
        optimizer.step() # Update model parameters.

        # Calculate and record performance metrics:
        total_loss += loss.item()
        num_batches += 1
        predictions = torch.argmax(output_batch, dim=1) # torch.argmax gets the predicted class index.
        correct = (predictions == target_batch).sum().item()
        total_correct += correct # The number of correct predictions.
        total_samples += target_batch.size(0) # The total number of samples.

    avg_loss = total_loss / num_batches
    accuracy = total_correct / total_samples * 100 # @Student: Calculate the accuracy by dividing total_correct by total_samples, and then multiplying by 100.
    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%, Time: {epoch_time:.2f}s")

total_time = time.time() - start_time
print(f"\nTotal training time: {total_time:.2f} seconds")

Epoch 1/20, Loss: 0.9600, Accuracy: 50.00%, Time: 0.20s
Epoch 2/20, Loss: 0.6188, Accuracy: 68.00%, Time: 0.06s
Epoch 3/20, Loss: 0.5624, Accuracy: 72.00%, Time: 0.06s
Epoch 4/20, Loss: 0.5269, Accuracy: 74.50%, Time: 0.07s
Epoch 5/20, Loss: 0.4974, Accuracy: 76.00%, Time: 0.07s
Epoch 6/20, Loss: 0.4729, Accuracy: 76.00%, Time: 0.07s
Epoch 7/20, Loss: 0.4497, Accuracy: 78.00%, Time: 0.07s
Epoch 8/20, Loss: 0.4296, Accuracy: 80.00%, Time: 0.07s
Epoch 9/20, Loss: 0.4078, Accuracy: 83.00%, Time: 0.07s
Epoch 10/20, Loss: 0.3895, Accuracy: 83.50%, Time: 0.07s
Epoch 11/20, Loss: 0.3694, Accuracy: 87.00%, Time: 0.07s
Epoch 12/20, Loss: 0.3494, Accuracy: 88.00%, Time: 0.08s
Epoch 13/20, Loss: 0.3305, Accuracy: 89.00%, Time: 0.08s
Epoch 14/20, Loss: 0.3154, Accuracy: 90.00%, Time: 0.08s
Epoch 15/20, Loss: 0.2941, Accuracy: 90.00%, Time: 0.08s
Epoch 16/20, Loss: 0.2727, Accuracy: 90.50%, Time: 0.08s
Epoch 17/20, Loss: 0.2492, Accuracy: 91.50%, Time: 0.08s
Epoch 18/20, Loss: 0.2253, Accuracy: 92.

The trained weights of the model are saved to disk so we can reuse them later without retraining.

In [11]:
os.makedirs(path_to_model, exist_ok=True)
torch.save(model.state_dict(), os.path.join(path_to_model, 'shape_model.pt'))

state_dict = torch.load(os.path.join(path_to_model, 'shape_model.pt'), weights_only=True)
model.load_state_dict(state_dict)
model.eval()

Sequential(
  (0): Linear(in_features=784, out_features=512, bias=True)
  (1): ReLU()
  (2): Linear(in_features=512, out_features=2, bias=True)
)

#### Model Testing and Visualization

In this section, the trained model is used to make predictions on test images. 

The labels corresponding to model output are defined and all test images are listed.

In [12]:
prediction_labels = ['triangle', 'circle']
test_images = os.listdir(path_to_test_images)

In [13]:
for picture in test_images:
    test_path = os.path.join(path_to_test_images, picture)
    test_image = preprocess_test_image(test_path) # Applies custom preprocessing.
    test_image_tensor = torch.tensor(test_image.flatten(), dtype=torch.float32).unsqueeze(0) # Converts to a tensor. 
    # Since the model expects 2D input ([batch_size, input_size]), unsqueeze(0) adds a batch dimension (shape becomes [1, input_size]).
    
    with torch.no_grad(): # Turns off gradient computation (saves memory, speeds up inference).
        prediction = model(test_image_tensor) # @Student: Feed test_image_tensor into the model to get logits (raw scores).
    
    # Apply softmax to get probabilities:
    probabilities = torch.softmax(prediction, dim=1)
    
    # Get predicted class and its probability
    predicted_class = torch.argmax(probabilities).item()  # Selects the index of the highest probability.
                                                        # This index corresponds to the predicted class label.
    probability = round(probabilities[0][predicted_class].item() * 100, 2)  # Retrieves the probability value for the predicted class, converts it to a percentage,
                                                                            # and rounds it to 2 decimal places for readability (e.g., 82.35%).
    
    # Displaying the image with the predicted label and probability:
    debug_image = cv.imread(test_path)
    resized_debug = cv.resize(debug_image, (512, 512))
    height, width, _ = resized_debug.shape
    debug_text = f"Prediction: {prediction_labels[predicted_class]} Probability: {probability}%"
    cv.putText(resized_debug, debug_text, (30, height - 50), cv.FONT_HERSHEY_SIMPLEX, 0.7, (209, 80, 0, 255), 2)
    cv.imshow("image", resized_debug)
    cv.waitKey(0)

size of resized test image:  (28, 28)


size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
size of resized test image:  (28, 28)
